# 🌍 City Quality of Life — Dataset Import

Downloads all datasets for the city comparison visualization project.

| Source type | Method | Used for |
|---|---|---|
| `kaggle` | `kagglehub` library | Main city-level datasets |
| `http` | Direct HTTP fetch | Single-file external CSVs |
| `numbeo_scrape` | `pandas.read_html` on Numbeo ranking pages | Country-level indices |
| `epi_scrape` | `pandas.read_html` on World Population Review | EF English Proficiency Index |

All files land in `data/raw/<folder>/`.

## 1. Setup — Install & Authenticate

In [1]:
import os
import time
import shutil
import requests
import pandas as pd
from pathlib import Path
import kagglehub

# -------------------------------------------------------
# 🔑 KAGGLE API TOKEN — paste your KGAT_... token here
# -------------------------------------------------------
os.environ["KAGGLE_API_TOKEN"] = "KGAT_xxxxxxxxxxxxxxxxxxxx"  # <-- replace
# -------------------------------------------------------

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)"}
SCRAPE_DELAY = 1.5  # seconds between scrape requests — be polite

print("✅ Token set")
print(f"📁 Data directory: {DATA_DIR.resolve()}")

✅ Token set
📁 Data directory: /Users/zenna/Library/CloudStorage/OneDrive-Mittuniversitetet/Visualization/Urban-Independence-Index/data/raw


/Users/zenna/miniconda3/envs/DT056A/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Dataset Registry

Each entry specifies `source`, `data_year`, `granularity`, and `description`.

- **Kaggle datasets** use `slug`
- **HTTP datasets** use `url` + `filename`
- **Numbeo scrape** uses a list of `pages` (each page = one index, scraped and merged)
- **EPI scrape** uses a single `url` pointing to World Population Review

In [ ]:
DATASETS = [

    # ── CORE — Kaggle ──────────────────────────────────────────────────────────
    {
        "name":        "Top Cities Quality of Life Index 2024",
        "source":      "kaggle",
        "slug":        "bilalabdulmalik/top-cities-worldwide-quality-of-life-index-2024",
        "folder":      "qol_index_2024",
        "category":    "core",
        "data_year":   "2024",
        "granularity": "city",
        "description": "Numbeo-sourced QoL index: purchasing power, safety, healthcare, "
                        "cost of living, pollution, and climate sub-scores."
    },
    {
        "name":        "Average Net Salary (World Cities)",
        "source":      "kaggle",
        "slug":        "cityapiio/average-net-salary-after-tax-main-world-cities",
        "folder":      "net_salary",
        "category":    "core",
        "data_year":   "2022",
        "granularity": "city",
        "description": "Average monthly net salary after tax for major world cities."
    },
    {
        "name":        "City Quality of Life Dataset",
        "source":      "kaggle",
        "slug":        "orhankaramancode/city-quality-of-life-dataset",
        "folder":      "qol_dataset",
        "category":    "core",
        "data_year":   "2021",
        "granularity": "city",
        "description": "City-level QoL scores across multiple dimensions (~200 cities, Numbeo-sourced)."
    },
    {
        "name":        "Global Cost of Living",
        "source":      "kaggle",
        "slug":        "mvieira101/global-cost-of-living",
        "folder":      "cost_of_living",
        "category":    "core",
        "data_year":   "2022",
        "granularity": "city",
        "description": "Detailed cost of living: rent, groceries, restaurants, transport, "
                        "utilities across 4,000+ cities."
    },

    # ── SUPPLEMENTARY — Kaggle ─────────────────────────────────────────────────
    {
        "name":        "World Happiness Report 2024",
        "source":      "kaggle",
        "slug":        "ajaypalsinghlo/world-happiness-report-2024",
        "folder":      "happiness_2024",
        "category":    "supplementary",
        "data_year":   "2024",
        "granularity": "country",
        "description": "Happiness ladder scores: GDP per capita, social support, healthy life "
                        "expectancy, freedom, generosity, corruption perception."
    },
    {
        "name":        "City Happiness Index 2024",
        "source":      "kaggle",
        "slug":        "emirhanai/city-happiness-index-2024",
        "folder":      "city_happiness_2024",
        "category":    "supplementary",
        "data_year":   "2024",
        "granularity": "city",
        "description": "City-level happiness index for 2024."
    },
    {
        "name":        "Sunshine Hours for Cities",
        "source":      "kaggle",
        "slug":        "bilalwaseer/sunshine-hours-for-cities-around-the-world",
        "folder":      "sunshine_hours",
        "category":    "supplementary",
        "data_year":   "2023",
        "granularity": "city",
        "description": "Annual and monthly sunshine hours per city — key lifestyle/climate factor."
    },
    {
        "name":        "World Cities Average Temperature",
        "source":      "kaggle",
        "slug":        "bilalwaseer/worlds-cities-with-their-average-temperature",
        "folder":      "avg_temperature",
        "category":    "supplementary",
        "data_year":   "2023",
        "granularity": "city",
        "description": "Average annual and monthly temperature per city worldwide."
    },
    {
        "name":        "Internet Download Speed 2024",
        "source":      "kaggle",
        "slug":        "chayanonc/internet-download-speed-comparison-2024",
        "folder":      "internet_speed_2024",
        "category":    "supplementary",
        "data_year":   "2024",
        "granularity": "country",
        "description": "Country-level broadband download speeds (Mbps) — proxy for digital infrastructure."
    },
    {
        "name":        "World Air Quality Index 2024",
        "source":      "kaggle",
        "slug":        "kanchana1990/world-air-quality-data-2024-updated",
        "folder":      "air_quality_2024",
        "category":    "supplementary",
        "data_year":   "2024",
        "granularity": "city",
        "description": "City-level Air Quality Index (AQI) and pollutant breakdown "
                        "(PM2.5, PM10, NO2, CO) for 2024."
    },

    # ── SUPPLEMENTARY — Direct HTTP ────────────────────────────────────────────
    {
        "name":        "LGBT+ Legal Equality Index (Equaldex via Our World in Data)",
        "source":      "http",
        "url":         "https://ourworldindata.org/grapher/lgbt-legal-equality-index.csv",
        "folder":      "lgbtq_index",
        "filename":    "lgbtq_legal_equality_index.csv",
        "category":    "supplementary",
        "data_year":   "2025",
        "granularity": "country",
        "description": "Country-level LGBT+ legal equality score (0–100) combining 13 legal "
                        "indicators: same-sex marriage, discrimination protections, gender "
                        "recognition, etc. Source: Equaldex via Our World in Data."
    },

    # ── SUPPLEMENTARY — Numbeo Country Scrape ─────────────────────────────────
    {
        "name":        "Numbeo Country Indices 2024 (Healthcare, Crime, Pollution)",
        "source":      "numbeo_scrape",
        "folder":      "numbeo_country_2024",
        "category":    "supplementary",
        "data_year":   "2024",
        "granularity": "country",
        "description": "Country-level Numbeo indices scraped directly from numbeo.com: "
                        "Healthcare Index, Crime Index, Safety Index, Pollution Index. "
                        "These complement the city-level QoL data already in the core datasets.",
        # Each page will be scraped, renamed, and merged into a single CSV
        "pages": [
            {
                "index_name": "healthcare",
                "url": "https://www.numbeo.com/health-care/rankings_by_country.jsp?title=2024"
            },
            {
                "index_name": "crime",
                "url": "https://www.numbeo.com/crime/rankings_by_country.jsp?title=2024"
            },
            {
                "index_name": "pollution",
                "url": "https://www.numbeo.com/pollution/rankings_by_country.jsp?title=2024"
            },
        ]
    },

    # ── SUPPLEMENTARY — EF EPI Scrape ─────────────────────────────────────────
    {
        "name":        "EF English Proficiency Index 2024",
        "source":      "epi_scrape",
        "url":         "https://worldpopulationreview.com/country-rankings/ef-english-proficiency-index-by-country",
        "folder":      "english_proficiency",
        "filename":    "ef_english_proficiency_index_2024.csv",
        "category":    "supplementary",
        "data_year":   "2024",
        "granularity": "country",
        "description": "Country-level EF English Proficiency Index scores (116 countries). "
                        "Scraped from World Population Review which republishes EF EPI data. "
                        "Relevant for young adults evaluating ease of integration abroad."
    },
]

print(f"📦 Total datasets    : {len(DATASETS)}")
print(f"   ├─ Core           : {sum(1 for d in DATASETS if d['category'] == 'core')}")
print(f"   └─ Supplementary  : {sum(1 for d in DATASETS if d['category'] == 'supplementary')}")
print()
print(f"  {'Name':<52} {'Year':<8} {'Grain':<10} {'Source'}")
print("  " + "-" * 85)
for d in DATASETS:
    print(f"  {d['name']:<52} {d['data_year']:<8} {d['granularity']:<10} {d['source']}")

📦 Total datasets    : 13
   ├─ Core           : 4
   └─ Supplementary  : 9

  Name                                                 Year     Grain      Source
  -------------------------------------------------------------------------------------
  Top Cities Quality of Life Index 2024                2024     city       kaggle
  Average Net Salary (World Cities)                    2022     city       kaggle
  City Quality of Life Dataset                         2021     city       kaggle
  Global Cost of Living                                2022     city       kaggle
  World Happiness Report 2024                          2024     country    kaggle
  City Happiness Index 2024                            2024     city       kaggle
  Sunshine Hours for Cities                            2023     city       kaggle
  World Cities Average Temperature                     2023     city       kaggle
  Internet Download Speed 2024                         2024     country    kaggle
  World Air Qual

## 3. Download Helpers

In [3]:
# ── Helper: Kaggle ────────────────────────────────────────────────────────────
def download_kaggle(dataset: dict, base_dir: Path) -> Path | None:
    dest = base_dir / dataset["folder"]
    dest.mkdir(parents=True, exist_ok=True)
    print(f"\n⬇️  [KAGGLE] {dataset['name']}  ({dataset['data_year']})")
    try:
        cache_path = Path(kagglehub.dataset_download(dataset["slug"]))
        copied = []
        for src in cache_path.rglob("*"):
            if src.is_file():
                shutil.copy2(src, dest / src.name)
                copied.append(src.name)
        print(f"   ✅ {len(copied)} file(s): {copied}")
        return dest
    except Exception as e:
        print(f"   ❌ {e}")
        return None


# ── Helper: Direct HTTP ───────────────────────────────────────────────────────
def download_http(dataset: dict, base_dir: Path) -> Path | None:
    dest = base_dir / dataset["folder"]
    dest.mkdir(parents=True, exist_ok=True)
    out  = dest / dataset["filename"]
    print(f"\n⬇️  [HTTP]   {dataset['name']}  ({dataset['data_year']})")
    try:
        r = requests.get(dataset["url"], headers=HEADERS, timeout=30)
        r.raise_for_status()
        out.write_bytes(r.content)
        print(f"   ✅ {out.name}  ({out.stat().st_size/1024:.1f} KB)")
        return dest
    except Exception as e:
        print(f"   ❌ {e}")
        return None


# ── Helper: Numbeo country scrape ─────────────────────────────────────────────
def scrape_numbeo_country(dataset: dict, base_dir: Path) -> Path | None:
    """
    Scrapes multiple Numbeo country ranking pages (one per index),
    standardises column names, and merges them into a single CSV
    joined on Country.
    """
    dest = base_dir / dataset["folder"]
    dest.mkdir(parents=True, exist_ok=True)
    print(f"\n⬇️  [NUMBEO] {dataset['name']}  ({dataset['data_year']})")

    merged = None
    for page in dataset["pages"]:
        time.sleep(SCRAPE_DELAY)
        try:
            tables = pd.read_html(
                requests.get(page["url"], headers=HEADERS, timeout=30).text
            )
            # Numbeo ranking tables are always the second table (index 1)
            df = tables[1].copy()

            # Standardise: drop Rank column, prefix other numeric cols with index name
            if "Rank" in df.columns:
                df = df.drop(columns=["Rank"])
            # Rename all non-Country columns to include the index prefix
            df.columns = [
                col if col == "Country"
                else f"{page['index_name']}_{col.replace(' ', '_').lower()}"
                for col in df.columns
            ]
            print(f"   ✅ {page['index_name']:12}  {len(df)} countries  "
                  f"cols: {list(df.columns)}")

            merged = df if merged is None else merged.merge(df, on="Country", how="outer")

        except Exception as e:
            print(f"   ❌ {page['index_name']}: {e}")

    if merged is not None:
        out = dest / "numbeo_country_indices_2024.csv"
        merged.to_csv(out, index=False)
        print(f"   💾 Saved: {out.name}  ({len(merged)} rows × {len(merged.columns)} cols)")
        return dest
    return None


# ── Helper: EF EPI scrape ─────────────────────────────────────────────────────
def scrape_epi(dataset: dict, base_dir: Path) -> Path | None:
    """
    Scrapes the EF English Proficiency Index table from World Population Review.
    """
    dest = base_dir / dataset["folder"]
    dest.mkdir(parents=True, exist_ok=True)
    print(f"\n⬇️  [EPI]    {dataset['name']}  ({dataset['data_year']})")
    try:
        time.sleep(SCRAPE_DELAY)
        tables = pd.read_html(
            requests.get(dataset["url"], headers=HEADERS, timeout=30).text
        )
        # Take the largest table — that's the country rankings
        df = max(tables, key=len).copy()
        out = dest / dataset["filename"]
        df.to_csv(out, index=False)
        print(f"   ✅ {out.name}  ({len(df)} rows × {len(df.columns)} cols)")
        print(f"   Columns: {list(df.columns)}")
        return dest
    except Exception as e:
        print(f"   ❌ {e}")
        return None


# ── Router ────────────────────────────────────────────────────────────────────
def download_dataset(dataset: dict, base_dir: Path) -> Path | None:
    dispatch = {
        "kaggle":        download_kaggle,
        "http":          download_http,
        "numbeo_scrape": scrape_numbeo_country,
        "epi_scrape":    scrape_epi,
    }
    fn = dispatch.get(dataset["source"])
    if fn is None:
        print(f"   ❌ Unknown source type: {dataset['source']}")
        return None
    return fn(dataset, base_dir)

## 4. Download All Datasets

In [4]:
print("=" * 70)
print("🚀 Downloading all datasets")
print("=" * 70)

results = {}
for dataset in DATASETS:
    path = download_dataset(dataset, DATA_DIR)
    results[dataset["folder"]] = {"path": path, "success": path is not None}

print("\n" + "=" * 70)
print("📋 Download Summary")
print("=" * 70)
print(f"  {'St':<4} {'Folder':<30} {'Year':<8} {'Grain':<10} {'Source'}")
print("  " + "-" * 68)
for ds in DATASETS:
    ok = results[ds["folder"]]["success"]
    print(f"  {'✅' if ok else '❌'}  {ds['folder']:<30} {ds['data_year']:<8} "
          f"{ds['granularity']:<10} {ds['source']}")

🚀 Downloading all datasets

⬇️  [KAGGLE] Top Cities Quality of Life Index 2024  (2024)
   ✅ 1 file(s): ['livable_cities.csv']

⬇️  [KAGGLE] Average Net Salary (World Cities)  (2022)
   ✅ 1 file(s): ['cities_average_net_salary_after_tax.19-10-2021.csv']

⬇️  [KAGGLE] City Quality of Life Dataset  (2021)
   ✅ 1 file(s): ['uaScoresDataFrame.csv']

⬇️  [KAGGLE] Global Cost of Living  (2022)
   ✅ 2 file(s): ['cost-of-living_v2.csv', 'cost-of-living.csv']

⬇️  [KAGGLE] World Happiness Report 2024  (2024)
   ✅ 1 file(s): ['WHR2024.csv']

⬇️  [KAGGLE] City Happiness Index 2024  (2024)
   ✅ 2 file(s): ['test.csv', 'train.csv']

⬇️  [KAGGLE] Sunshine Hours for Cities  (2023)
   ✅ 1 file(s): ['Sunshine hours for cities in the world.csv']

⬇️  [KAGGLE] World Cities Average Temperature  (2023)
   ✅ 1 file(s): ['worlds all cities with their avg temp - Sheet1.csv']

⬇️  [KAGGLE] Internet Download Speed 2024  (2024)
   ✅ 2 file(s): ['internet_speed_by_city_may2024.csv', 'internet_speed_by_country_may2

100%|██████████| 1.64M/1.64M [00:09<00:00, 180kB/s]

Extracting files...


   ✅ 1 file(s): ['world_air_quality.csv']

⬇️  [HTTP]   LGBT+ Legal Equality Index (Equaldex via Our World in Data)  (2025)
   ✅ lgbtq_legal_equality_index.csv  (4.8 KB)

⬇️  [NUMBEO] Numbeo Country Indices 2024 (Healthcare, Crime, Pollution)  (2024)


/var/folders/vx/wsv0__k15g37xmltv38_43l80000gn/T/ipykernel_51750/2538814998.py:52: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(


...

/var/folders/vx/wsv0__k15g37xmltv38_43l80000gn/T/ipykernel_51750/2538814998.py:52: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(


   ✅ crime         146 countries  cols: ['Country', 'crime_crime_index', 'crime_safety_index']


/var/folders/vx/wsv0__k15g37xmltv38_43l80000gn/T/ipykernel_51750/2538814998.py:52: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(


   ✅ pollution     112 countries  cols: ['Country', 'pollution_pollution_index', 'pollution_exp_pollution_index']
   💾 Saved: numbeo_country_indices_2024.csv  (146 rows × 7 cols)

⬇️  [EPI]    EF English Proficiency Index 2024  (2024)
   ✅ ef_english_proficiency_index_2024.csv  (116 rows × 4 cols)
   Columns: ['Unnamed: 0', 'Country', 'English Proficiency Index 2025 (0-800)â\x86\x93', 'EF Proficiency Band 2025']

📋 Download Summary
  St   Folder                         Year     Grain      Source
  --------------------------------------------------------------------
  ✅  qol_index_2024                 2024     city       kaggle
  ✅  net_salary                     2022     city       kaggle
  ✅  qol_dataset                    2021     city       kaggle
  ✅  cost_of_living                 2022     city       kaggle
  ✅  happiness_2024                 2024     country    kaggle
  ✅  city_happiness_2024            2024     city       kaggle
  ✅  sunshine_hours                 2023     city 

/var/folders/vx/wsv0__k15g37xmltv38_43l80000gn/T/ipykernel_51750/2538814998.py:93: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(


## 5. Preview All Datasets

In [5]:
def preview_dataset(dataset: dict, base_dir: Path, n_rows: int = 3):
    folder    = base_dir / dataset["folder"]
    csv_files = list(folder.glob("*.csv")) if folder.exists() else []

    print(f"\n{'─'*70}")
    print(f"📂 {dataset['name']}")
    print(f"   Category    : {dataset['category'].upper()}  |  Source: {dataset['source']}  |  "
          f"Year: {dataset['data_year']}  |  Granularity: {dataset['granularity']}")
    print(f"   Description : {dataset['description']}")

    if not csv_files:
        all_files = list(folder.iterdir()) if folder.exists() else []
        print(f"   ⚠️  No CSVs found. Files: {[f.name for f in all_files]}")
        return None

    dfs = {}
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file)
            dfs[csv_file.name] = df
            print(f"\n   📄 {csv_file.name}  —  {df.shape[0]:,} rows × {df.shape[1]} cols")
            print(f"   Columns: {list(df.columns)}")
            display(df.head(n_rows))
        except Exception as e:
            print(f"   ❌ {csv_file.name}: {e}")
    return dfs


print("=" * 70)
print("👀 Previewing all datasets")
print("=" * 70)

all_dfs = {}
for dataset in DATASETS:
    dfs = preview_dataset(dataset, DATA_DIR)
    if dfs:
        all_dfs[dataset["folder"]] = dfs

👀 Previewing all datasets

──────────────────────────────────────────────────────────────────────
📂 Top Cities Quality of Life Index 2024
   Category    : CORE  |  Source: kaggle  |  Year: 2024  |  Granularity: city
   Description : Numbeo-sourced QoL index: purchasing power, safety, healthcare, cost of living, pollution, and climate sub-scores.

   📄 livable_cities.csv  —  150 rows × 12 cols
   Columns: ['Rank', 'City', 'Country', 'Quality of Life Index', 'Purchasing Power Index', 'Safety Index', 'Health Care Index', 'Cost of Living Index', 'Property Price to Income Ratio', 'Traffic Commute Time Index', 'Pollution Index', 'Climate Index']


,Rank,City,Country,Quality of Life Index,Purchasing Power Index,Safety Index,Health Care Index,Cost of Living Index,Property Price to Income Ratio,Traffic Commute Time Index,Pollution Index,Climate Index
0,1,The Hague,Netherlands,223.8,138.9,79.8,80.7,60.2,5.9,21.0,17.5,90.6
1,2,Luxembourg,Luxembourg,220.6,175.9,71.4,76.6,62.0,9.2,27.7,21.6,82.6
2,3,Eindhoven,Netherlands,217.5,139.4,78.1,78.9,62.2,6.0,24.5,19.2,85.4



──────────────────────────────────────────────────────────────────────
📂 Average Net Salary (World Cities)
   Category    : CORE  |  Source: kaggle  |  Year: 2022  |  Granularity: city
   Description : Average monthly net salary after tax for major world cities.

   📄 cities_average_net_salary_after_tax.19-10-2021.csv  —  685 rows × 25 cols
   Columns: ['City', ' "Region"', ' "Country"', ' "Average Monthly Net Salary (After Tax)', ' 2010"', ' "Average Monthly Net Salary (After Tax).1', ' 2011"', ' "Average Monthly Net Salary (After Tax).2', ' 2012"', ' "Average Monthly Net Salary (After Tax).3', ' 2013"', ' "Average Monthly Net Salary (After Tax).4', ' 2014"', ' "Average Monthly Net Salary (After Tax).5', ' 2015"', ' "Average Monthly Net Salary (After Tax).6', ' 2016"', ' "Average Monthly Net Salary (After Tax).7', ' 2017"', ' "Average Monthly Net Salary (After Tax).8', ' 2018"', ' "Average Monthly Net Salary (After Tax).9', ' 2019"', ' "Average Monthly Net Salary (After Tax).10', ' 2

,City,"""Region""","""Country""","""Average Monthly Net Salary (After Tax)","2010""","""Average Monthly Net Salary (After Tax).1","2011""","""Average Monthly Net Salary (After Tax).2","2012""","""Average Monthly Net Salary (After Tax).3",...,"""Average Monthly Net Salary (After Tax).6","2016""","""Average Monthly Net Salary (After Tax).7","2017""","""Average Monthly Net Salary (After Tax).8","2018""","""Average Monthly Net Salary (After Tax).9","2019""","""Average Monthly Net Salary (After Tax).10","2020"""
0,New York City,"""New York""","""United States of America""",3300.0,3568.530000,4017.991667,4339.513069,3476.340323,3376.010678,4026.656605,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Washington, D.C.","""District of Columbia""","""United States of America""",2200.0,3365.200744,3700.000000,3981.700000,4573.356667,3479.050000,3900.222222,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,San Francisco,"""California""","""United States of America""",4700.0,2633.333333,4195.428571,4167.538462,3470.000000,4058.783333,4281.510400,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



──────────────────────────────────────────────────────────────────────
📂 City Quality of Life Dataset
   Category    : CORE  |  Source: kaggle  |  Year: 2021  |  Granularity: city
   Description : City-level QoL scores across multiple dimensions (~200 cities, Numbeo-sourced).

   📄 uaScoresDataFrame.csv  —  266 rows × 21 cols
   Columns: ['Unnamed: 0', 'UA_Name', 'UA_Country', 'UA_Continent', 'Housing', 'Cost of Living', 'Startups', 'Venture Capital', 'Travel Connectivity', 'Commute', 'Business Freedom', 'Safety', 'Healthcare', 'Education', 'Environmental Quality', 'Economy', 'Taxation', 'Internet Access', 'Leisure & Culture', 'Tolerance', 'Outdoors']


...

,Entity,Code,Year,Legal equality index
0,Afghanistan,AFG,2025,0.510000
1,Africa,OWID_AFR,2025,17.871979
2,Albania,ALB,2025,62.270000



──────────────────────────────────────────────────────────────────────
📂 Numbeo Country Indices 2024 (Healthcare, Crime, Pollution)
   Category    : SUPPLEMENTARY  |  Source: numbeo_scrape  |  Year: 2024  |  Granularity: country
   Description : Country-level Numbeo indices scraped directly from numbeo.com: Healthcare Index, Crime Index, Safety Index, Pollution Index. These complement the city-level QoL data already in the core datasets.

   📄 numbeo_country_indices_2024.csv  —  146 rows × 7 cols
   Columns: ['Country', 'healthcare_health_care_index', 'healthcare_health_care_exp._index', 'crime_crime_index', 'crime_safety_index', 'pollution_pollution_index', 'pollution_exp_pollution_index']


,Country,healthcare_health_care_index,healthcare_health_care_exp._index,crime_crime_index,crime_safety_index,pollution_pollution_index,pollution_exp_pollution_index
0,Afghanistan,NaN,NaN,78.3,21.7,84.9,151.5
1,Albania,49.5,86.1,45.6,54.4,77.0,135.0
2,Algeria,54.7,99.0,52.2,47.8,63.2,109.9



──────────────────────────────────────────────────────────────────────
📂 EF English Proficiency Index 2024
   Category    : SUPPLEMENTARY  |  Source: epi_scrape  |  Year: 2024  |  Granularity: country
   Description : Country-level EF English Proficiency Index scores (116 countries). Scraped from World Population Review which republishes EF EPI data. Relevant for young adults evaluating ease of integration abroad.

   📄 ef_english_proficiency_index_2024.csv  —  116 rows × 4 cols
   Columns: ['Unnamed: 0', 'Country', 'English Proficiency Index 2025 (0-800)â\x86\x93', 'EF Proficiency Band 2025']


,Unnamed: 0,Country,English Proficiency Index 2025 (0-800)â,EF Proficiency Band 2025
0,NaN,Netherlands,636,Very high
1,NaN,Norway,610,Very high
2,NaN,Singapore,609,Very high


## 6. File Structure Overview

In [6]:
print("📁 data/raw/ file tree:\n")
for ds in DATASETS:
    folder = DATA_DIR / ds["folder"]
    cat    = "[CORE]" if ds["category"] == "core" else "[SUPP]"
    print(f"  {cat} {ds['folder']}/  (year: {ds['data_year']}, {ds['granularity']}-level, source: {ds['source']})")
    if folder.exists():
        for f in sorted(folder.iterdir()):
            print(f"        ├─ {f.name}  ({f.stat().st_size/1024:.1f} KB)")
    else:
        print(f"        └─ ⚠️  folder not found")
    print()

📁 data/raw/ file tree:

  [CORE] qol_index_2024/  (year: 2024, city-level, source: kaggle)
        ├─ livable_cities.csv  (13.3 KB)

  [CORE] net_salary/  (year: 2022, city-level, source: kaggle)
        ├─ cities_average_net_salary_after_tax.19-10-2021.csv  (106.3 KB)

  [CORE] qol_dataset/  (year: 2021, city-level, source: kaggle)
        ├─ uaScoresDataFrame.csv  (61.0 KB)

  [CORE] cost_of_living/  (year: 2022, city-level, source: kaggle)
        ├─ cost-of-living.csv  (1371.5 KB)
        ├─ cost-of-living_v2.csv  (1508.7 KB)

  [SUPP] happiness_2024/  (year: 2024, country-level, source: kaggle)
        ├─ WHR2024.csv  (10.0 KB)

  [SUPP] city_happiness_2024/  (year: 2024, city-level, source: kaggle)
        ├─ test.csv  (2.4 KB)
        ├─ train.csv  (26.6 KB)

  [SUPP] sunshine_hours/  (year: 2023, city-level, source: kaggle)
        ├─ Sunshine hours for cities in the world.csv  (37.1 KB)

  [SUPP] avg_temperature/  (year: 2023, city-level, source: kaggle)
        ├─ worlds all 

---
✅ **All datasets downloaded.** Next step: data cleaning & merging.

### Full dataset inventory

| # | Name | Year | Granularity | Category | Source |
|---|------|------|-------------|----------|--------|
| 1 | Top Cities QoL Index | 2024 | City | Core | Kaggle |
| 2 | Average Net Salary | 2022 | City | Core | Kaggle |
| 3 | City QoL Dataset | 2021 | City | Core | Kaggle |
| 4 | Global Cost of Living | 2022 | City | Core | Kaggle |
| 5 | World Happiness Report | 2024 | Country | Supplementary | Kaggle |
| 6 | City Happiness Index | 2024 | City | Supplementary | Kaggle |
| 7 | Sunshine Hours | 2023 | City | Supplementary | Kaggle |
| 8 | Average Temperature | 2023 | City | Supplementary | Kaggle |
| 9 | Internet Speed | 2024 | Country | Supplementary | Kaggle |
| 10 | World Air Quality Index | 2024 | City | Supplementary | Kaggle |
| 11 | LGBT+ Legal Equality Index | 2025 | Country | Supplementary | HTTP (OWID/Equaldex) |
| 12 | Numbeo: Healthcare / Crime / Pollution | 2024 | Country | Supplementary | Scrape (Numbeo) |
| 13 | EF English Proficiency Index | 2024 | Country | Supplementary | Scrape (WPR/EF EPI) |